# PCMCI+ Temporal Causal Discovery Analysis

This notebook runs PCMCI+ across tau values for 10 deterministic ER-neuron subsets sampled from the same recording.

## Objective
- Sample 10 ER subsets using fixed seeds
- Draw subset size k uniformly from 20 to 30 for each seed
- Keep at most 35% of sampled neurons from EM and the remainder from REC
- Remove duplicate indices by sampling without replacement
- Preserve the provided ordering of selected indices
- Save adjacency matrices for each tau value

## Configuration
- **Trials per tau**: 10
- **Tau range**: 1 to 7 (maximum lag to consider)
- **Subset size range**: 20 to 30 neurons
- **EM cap**: up to 35% of the sampled subset
- **Method**: PCMCI+ (Tigramite implementation)

In [1]:
# Setup and imports
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import sys

# Locate project root
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "src" / "markovianity_diagnostic").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root")

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from markovianity_diagnostic.core.utils import adj_mtx,  continuous_noise_fun

# Import PCMCI+ from tigramite
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

print("✓ All imports successful")

✓ All imports successful


In [2]:
# Configuration for deterministic multi-run ER sampling
NUM_TRIALS = 10
SEEDS = [12, 42, 10, 11, 5, 132, 52, 71, 11, 20]
np.random.seed(SEEDS[0])

# PCMCI+ hyperparameters
TAU_MIN = 1
TAU_MAX = 7
TAU_RANGE = list(range(TAU_MIN, TAU_MAX + 1))
PC_ALPHA = 0.05  # Significance level for independence tests
SAMPLE_SIZE_RANGE = (15, 25)

# Neuron groups copied from Using_whole_data_ER.ipynb
EM_idx = [54, 55, 61, 130, 133, 142, 154, 193, 194, 195, 275, 276, 280, 290, 294, 298, 453, 471, 501, 619, 622, 642, 644, 693, 699, 700, 704, 721, 968, 984, 999]
REC_idx = [4, 12, 14, 30, 34, 36, 70, 73, 77, 96, 97, 102, 103, 217, 227, 228, 235, 245, 246, 251, 257, 310, 317, 337, 348, 352, 365, 388, 405, 411, 416, 427, 432, 433, 435, 472, 473, 478, 498, 521, 540, 541, 542, 570, 571, 575, 580, 617, 620, 647, 651, 665, 675, 717, 755, 764, 770, 796, 798, 801, 807, 810, 823, 824, 828, 829, 881, 886, 928]
ER_ORDER = EM_idx + REC_idx

# Input: use the provided NumPy fluorescence signals (neurons x frames)
INPUT_NPY = PROJECT_ROOT / 'data' / 'v2a-RSNs' / '220119_F2_run11' / '220119_F2_F2_run11_cells_fluorescence_signals.npy'

# Output: save adjacency dicts per seed
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'pcmciplus'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Trials per tau: {NUM_TRIALS}")
print(f"  Seeds: {SEEDS}")
print(f"  Tau range: {TAU_RANGE}")
print(f"  Sample size range: {SAMPLE_SIZE_RANGE}")
print(f"  PC alpha: {PC_ALPHA}")
print(f"  Input file: {INPUT_NPY}")
print(f"  Output dir: {OUTPUT_DIR}")

Configuration:
  Trials per tau: 10
  Seeds: [12, 42, 10, 11, 5, 132, 52, 71, 11, 20]
  Tau range: [1, 2, 3, 4, 5, 6, 7]
  Sample size range: (15, 25)
  PC alpha: 0.05
  Input file: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/data/v2a-RSNs/220119_F2_run11/220119_F2_F2_run11_cells_fluorescence_signals.npy
  Output dir: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/v2a-RSNs/pcmciplus


In [3]:
def recovery_metrics(predicted: np.ndarray, truth: np.ndarray) -> dict[str, float]:
    """Compute accuracy, precision, recall, FPR, balanced_accuracy, F1 for binary adjacencies."""
    predicted = (predicted > 0).astype(int)
    truth = (truth > 0).astype(int)

    # Zero diagonals
    np.fill_diagonal(predicted, 0)
    np.fill_diagonal(truth, 0)

    tp = int(np.logical_and(predicted == 1, truth == 1).sum())
    tn = int(np.logical_and(predicted == 0, truth == 0).sum())
    fp = int(np.logical_and(predicted == 1, truth == 0).sum())
    fn = int(np.logical_and(predicted == 0, truth == 1).sum())

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = (specificity + recall) / 2
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'fpr': float(fpr),
        'balanced_accuracy': float(balanced_accuracy),
        'f1': float(f1),
        'tp': float(tp),
        'fp': float(fp),
        'tn': float(tn),
        'fn': float(fn),
    }

def graph_to_adjacency(causal_graph) -> np.ndarray:
    """Convert a Tigramite PCMCI+ graph to a binary adjacency matrix.

    Any directed edge at any lag is treated as present in the collapsed adjacency.
    """
    n_vars = causal_graph.shape[0]
    adjacency = np.zeros((n_vars, n_vars), dtype=int)

    for source in range(causal_graph.shape[0]):
        for target in range(causal_graph.shape[1]):
            for lag in range(causal_graph.shape[2]):
                edge_type = causal_graph[source, target, lag]
                if isinstance(edge_type, str) and '>' in edge_type and '<' not in edge_type:
                    adjacency[target, source] = 1

    return adjacency

print("✓ Helper functions defined")

✓ Helper functions defined


In [4]:
# Run PCMCI+ for 10 deterministic ER subsets sampled from the same recording.
# Data shape: (neurons, frames). Remove bad frames, then sample ER neurons without duplicates.
import pickle
import pandas as pd


def sample_er_indices(seed: int) -> tuple[int, np.ndarray, int, int]:
    """Sample a deterministic ER subset from EM_idx and REC_idx."""
    rng = np.random.default_rng(seed)
    k = int(rng.integers(SAMPLE_SIZE_RANGE[0], SAMPLE_SIZE_RANGE[1] + 1))
    n_em = min(len(EM_idx), int(np.floor(0.35 * k)))
    n_rec = k - n_em

    em_sample = rng.choice(EM_idx, size=n_em, replace=False) if n_em else np.array([], dtype=int)
    rec_sample = rng.choice(REC_idx, size=n_rec, replace=False) if n_rec else np.array([], dtype=int)

    sampled_set = {int(idx) for idx in np.concatenate([em_sample, rec_sample])}
    sampled_ordered = np.array([idx for idx in ER_ORDER if idx in sampled_set], dtype=int)
    return k, sampled_ordered, n_em, n_rec


data = np.load(INPUT_NPY)
print('Loaded data shape (neurons, frames):', data.shape)

# Bad frames to remove (from Using_whole_data_ER.ipynb)
bad_frames = [13, 87, 88, 975, 1253, 1555, 1556, 1700, 1701, 1761, 2268, 3003]

# Remove bad frames from axis=1 (frames)
new_traces = np.delete(data, bad_frames, axis=1)
print('After removing bad frames shape (neurons, frames):', new_traces.shape)

run_manifest = []
adjacency_runs = {}
out_name = INPUT_NPY.parent.name  # e.g., '220119_F2_run11'

for run_id, seed in enumerate(SEEDS, start=1):
    k, ER_idx, n_em, n_rec = sample_er_indices(seed)
    X = new_traces[ER_idx]
    print(f'Run {run_id}/{NUM_TRIALS} seed={seed}: k={k}, EM={n_em}, REC={n_rec}, X shape={X.shape}')

    # Impute any remaining NaNs (interpolate then fill with column mean)
    data_df = pd.DataFrame(X.T)  # (frames, neurons)
    data_df = data_df.interpolate(axis=0, limit_direction='both')
    data_df = data_df.fillna(data_df.mean())
    X = data_df.values.T  # back to (neurons, frames)

    seed_adjacency = {}
    for tau in TAU_RANGE:
        print(f"  Running PCMCI+ tau={tau} on {X.shape[0]} variables")
        dataframe = pp.DataFrame(X.T)
        pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=0)
        results_pcmci = pcmci.run_pcmciplus(tau_max=tau, pc_alpha=PC_ALPHA)

        causal_graph = results_pcmci['graph']
        adjacency_pred = np.zeros((X.shape[0], X.shape[0]), dtype=bool)
        for source in range(causal_graph.shape[0]):
            for target in range(causal_graph.shape[1]):
                for lag in range(causal_graph.shape[2]):
                    edge_type = causal_graph[source, target, lag]
                    if isinstance(edge_type, str) and '>' in edge_type and '<' not in edge_type:
                        adjacency_pred[target, source] = True

        np.fill_diagonal(adjacency_pred, False)
        seed_adjacency[int(tau)] = adjacency_pred
        print(f"    tau={tau} adjacency shape: {adjacency_pred.shape}, edges: {adjacency_pred.sum()}")

    adjacency_runs[int(seed)] = seed_adjacency
    run_manifest.append({
        'run_id': run_id,
        'seed': int(seed),
        'k': int(k),
        'n_em': int(n_em),
        'n_rec': int(n_rec),
        'er_idx': ER_idx.tolist(),
    })

    out_path = OUTPUT_DIR / f"{out_name}_seed{seed}.pkl"
    with open(out_path, 'wb') as f:
        pickle.dump(seed_adjacency, f)
    print(f"  Saved adjacency dict to {out_path}")

manifest_path = OUTPUT_DIR / f"{out_name}_sampling_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)

combined_path = OUTPUT_DIR / f"{out_name}_all_seeds.pkl"
with open(combined_path, 'wb') as f:
    pickle.dump(adjacency_runs, f)

print(f"Saved sampling manifest to {manifest_path}")
print(f"Saved combined adjacency runs to {combined_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/data/v2a-RSNs/220119_F2_run11/220119_F2_F2_run11_cells_fluorescence_signals.npy'

In [ ]:
from markovianity_diagnostic.experiments.graph_metrics import compute_graph_stability_metrics


def _mean_std(values: list[float]) -> dict[str, float]:
    array = np.asarray(values, dtype=float)
    return {
        'mean': float(np.mean(array)),
        'std': float(np.std(array)),
    }


if not adjacency_runs:
    raise ValueError('No adjacency runs were produced; cannot compute metrics.')

seed_metrics = {
    int(seed): compute_graph_stability_metrics(seed_adjacency)
    for seed, seed_adjacency in adjacency_runs.items()
}
seed_ids = sorted(seed_metrics)
tau_values = sorted(next(iter(adjacency_runs.values())).keys())
reference_metrics = next(iter(seed_metrics.values()))

seed_rows = []
for seed in seed_ids:
    metrics = seed_metrics[seed]
    for tau in tau_values:
        row = {
            'seed': seed,
            'tau': int(tau),
            'edge_count': int(metrics['edge_counts'][tau]),
            'T_obs': float(metrics['T_obs']),
        }
        if tau in metrics['D_p']:
            row['D_p'] = float(metrics['D_p'][tau])
            row['D_minus'] = float(metrics['D_parts'][tau]['D_minus'])
            row['D_plus'] = float(metrics['D_parts'][tau]['D_plus'])
        else:
            row['D_p'] = np.nan
            row['D_minus'] = np.nan
            row['D_plus'] = np.nan
        seed_rows.append(row)

seed_metrics_df = pd.DataFrame(seed_rows).sort_values(['seed', 'tau']).reset_index(drop=True)
seed_metrics_csv = OUTPUT_DIR / f'{out_name}_seed_metrics.csv'
seed_metrics_df.to_csv(seed_metrics_csv, index=False)

summary_by_tau_rows = []
for tau in tau_values:
    edge_values = [seed_metrics[seed]['edge_counts'][tau] for seed in seed_ids]
    row = {
        'tau': int(tau),
        'edge_count_mean': _mean_std(edge_values)['mean'],
        'edge_count_std': _mean_std(edge_values)['std'],
        'T_obs_mean': _mean_std([seed_metrics[seed]['T_obs'] for seed in seed_ids])['mean'],
        'T_obs_std': _mean_std([seed_metrics[seed]['T_obs'] for seed in seed_ids])['std'],
    }
    if tau in reference_metrics['D_p']:
        d_p_values = [seed_metrics[seed]['D_p'][tau] for seed in seed_ids]
        d_minus_values = [seed_metrics[seed]['D_parts'][tau]['D_minus'] for seed in seed_ids]
        d_plus_values = [seed_metrics[seed]['D_parts'][tau]['D_plus'] for seed in seed_ids]
        row.update({
            'D_p_mean': _mean_std(d_p_values)['mean'],
            'D_p_std': _mean_std(d_p_values)['std'],
            'D_minus_mean': _mean_std(d_minus_values)['mean'],
            'D_minus_std': _mean_std(d_minus_values)['std'],
            'D_plus_mean': _mean_std(d_plus_values)['mean'],
            'D_plus_std': _mean_std(d_plus_values)['std'],
        })
    else:
        row.update({
            'D_p_mean': np.nan,
            'D_p_std': np.nan,
            'D_minus_mean': np.nan,
            'D_minus_std': np.nan,
            'D_plus_mean': np.nan,
            'D_plus_std': np.nan,
        })
    summary_by_tau_rows.append(row)

summary_by_tau_df = pd.DataFrame(summary_by_tau_rows).sort_values('tau').reset_index(drop=True)
summary_by_tau_csv = OUTPUT_DIR / f'{out_name}_summary_by_tau.csv'
summary_by_tau_df.to_csv(summary_by_tau_csv, index=False)

recording_summary = {
    'dataset': out_name,
    'file': combined_path.name,
    'n_nodes': int(next(iter(next(iter(adjacency_runs.values())).values())).shape[0]),
    'n_p_values': len(tau_values),
    'n_seeds': len(seed_ids),
    'T_obs': _mean_std([seed_metrics[seed]['T_obs'] for seed in seed_ids]),
    'edge_counts': {
        str(tau): _mean_std([seed_metrics[seed]['edge_counts'][tau] for seed in seed_ids])
        for tau in tau_values
    },
    'D_p': {
        str(tau): _mean_std([seed_metrics[seed]['D_p'][tau] for seed in seed_ids])
        for tau in tau_values
        if tau in reference_metrics['D_p']
    },
    'D_parts': {
        str(tau): {
            'D_minus': _mean_std([seed_metrics[seed]['D_parts'][tau]['D_minus'] for seed in seed_ids]),
            'D_plus': _mean_std([seed_metrics[seed]['D_parts'][tau]['D_plus'] for seed in seed_ids]),
        }
        for tau in tau_values
        if tau in reference_metrics['D_p']
    },
}

summary_csv = OUTPUT_DIR / 'summary.csv'
pd.DataFrame([recording_summary]).to_csv(summary_csv, index=False)

summary_json = OUTPUT_DIR / 'summary.json'
with open(summary_json, 'w', encoding='utf-8') as f:
    json.dump(recording_summary, f, indent=2)

print(f'Saved seed metrics to {seed_metrics_csv}')
print(f'Saved per-tau summary to {summary_by_tau_csv}')
print(f'Saved recording summary to {summary_csv}')
print(f'Saved recording summary JSON to {summary_json}')